# Module 11 — Chọn đúng kiểm định thống kê (ôn tập tổng hợp)
Notebook RIÊNG cho Module 11 — chạy lại 6 tình huống nghiên cứu thật đã xác định ở trang web, mỗi tình huống dùng ĐÚNG kiểm định đã chọn theo cây quyết định. Không có phân tích thống kê MỚI — đây là notebook ôn tập, tổng hợp lại syntax đã học ở Module 05-10.

In [ ]:
# Chạy ô này đầu tiên (Colab: bấm ▶). Không cần cài gì thêm.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """Bộ dữ liệu GIẢ LẬP 'Lớp học 240 học sinh' — CÙNG một seed=2026 dùng
    xuyên suốt các module 03-08, khớp chính xác với lop_hoc_240.csv/.sav
    trong repo (không phải dữ liệu thật, nhưng tái lập được 100%)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Nam", "Nữ"], n)
    method = rng.choice(["Truyền thống", "Dự án"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Dự án")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Dự án") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("Đã tạo df:", df.shape)

In [ ]:
df["gain"] = df["posttest"] - df["pretest"]

## Tình huống 1 — So sánh 3 trường ĐỘC LẬP, biến tham số → ANOVA một chiều (Module 06)

In [ ]:
groups = [g["math"].values for _, g in df.groupby("school")]
f_stat, p_val = stats.f_oneway(*groups)
print(f"F={f_stat:.3f}, p={p_val:.4f}")

## Tình huống 2 — Cùng đối tượng, đo 2 lần, tham số → T-test bắt cặp (Module 05)

In [ ]:
t, p = stats.ttest_rel(df["posttest"], df["pretest"])
print(f"t={t:.2f}, p={p:.2e}")

## Tình huống 3 — 3 nhóm độc lập, biến thứ bậc (Likert) → Kruskal-Wallis (Module 07)

In [ ]:
groups_h1 = [g["h1"].dropna().values for _, g in df.groupby("school")]
h_stat, p_kw = stats.kruskal(*groups_h1)
print(f"H={h_stat:.3f}, p={p_kw:.3f}")

## Tình huống 4 — Đo liên hệ (Pearson r) RỒI dự đoán (hồi quy) (Module 08 + 09)

In [ ]:
mask = df["study_hours"] > 20
df.loc[mask, "study_hours"] = df.loc[mask, "study_hours"] / 10
r, p_r = stats.pearsonr(df["study_hours"], df["math"])
print(f"Bước 1 — Pearson r: r={r:.3f}, p={p_r:.2e}")

import statsmodels.api as sm
X = sm.add_constant(df[["study_hours"]])
m = sm.OLS(df["math"], X).fit()
print(f"Bước 2 — Hồi quy: math = {m.params['const']:.2f} + {m.params['study_hours']:.2f}×study_hours, R²={m.rsquared:.3f}")

## Tình huống 5 — Tìm cấu trúc ẩn từ NHIỀU BIẾN → Phân tích nhân tố (Module 10)

In [ ]:
items = df[["h1","h2","h3","a1","a2","a3"]].dropna()
from factor_analyzer import FactorAnalyzer
fa = FactorAnalyzer(n_factors=2, rotation="varimax")
fa.fit(items)
print(pd.DataFrame(fa.loadings_, index=items.columns, columns=["NT1","NT2"]).round(3))

## Tình huống 6 — Nhóm các ĐỐI TƯỢNG (trường) có hồ sơ giống nhau → Phân tích cụm (Module 10)

In [ ]:
RAW = "https://raw.githubusercontent.com/TatcataiTTN/for-Social-Science/main/SPSS/data/sav/"
truong = pd.read_csv(RAW + "vnm_truong_195_tong_hop.csv")
vars_ = ["EDULEAD","NEGSCLIM","STAFFSHORT","EDUSHORT","DIGPREP"]
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
d = truong[vars_].dropna()
X = StandardScaler().fit_transform(d)
km = KMeans(n_clusters=3, random_state=2026, n_init=10).fit(X)
print("Cỡ mỗi cụm:", list(np.bincount(km.labels_)))

## Tổng kết: bảng tra nhanh 6 tình huống
| # | Câu hỏi nghiên cứu | Phương pháp đã chọn | Module |
|---|---|---|---|
| 1 | 3 trường độc lập, điểm Toán (tham số) | ANOVA một chiều | 06 |
| 2 | Cùng học sinh, pretest→posttest | T-test bắt cặp | 05 |
| 3 | 3 trường độc lập, đánh giá Likert | Kruskal-Wallis | 07 |
| 4 | Liên hệ VÀ dự đoán | Pearson r + Hồi quy | 08, 09 |
| 5 | Cấu trúc ẩn từ 6 câu hỏi | Phân tích nhân tố (EFA) | 10 |
| 6 | Nhóm 195 trường theo hồ sơ | Phân tích cụm (K-means) | 10 |

Đây chính là cây quyết định 4 câu hỏi đã trình bày trên trang web — áp dụng đúng thứ tự (mục tiêu → thang đo → số nhóm/hướng liên hệ) sẽ luôn dẫn tới đúng hàng trong bảng này.